<a href="https://colab.research.google.com/github/emmanuelle-bul/AI_Exercises/blob/main/FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [69]:
!pip install -q chromadb openai pypdf

In [70]:
from pathlib import Path
from pypdf import PdfReader
from typing import List
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from google.colab import userdata
from openai import OpenAI
from openai.types.responses import Response
from google.colab import userdata
from pydantic import BaseModel, Field

In [71]:
import re, json

In [72]:
openai_api_key = userdata.get('OPENAI_API_KEY')
openai_client = OpenAI(api_key=openai_api_key)

chroma_client = PersistentClient("/content/chromadb")
#chroma_client.delete_collection(name="final_project")
chroma_collection = chroma_client.get_or_create_collection(
    name="final_project",
    embedding_function=OpenAIEmbeddingFunction(api_key=openai_api_key, model_name="text-embedding-3-large")
)

# ако искаме всеки път да получаваме празна DB напр. при експериментиране с много .pdf файлове и не искаме те да си пречат
#  => delete_collection + create_collection
# chroma_collection.delete(ids=sentences_ids)

# Изчитане на текста от .pdf

In [73]:
# функция, на която подаваме път към .pdf документа, изчита съдържанието на файла и ни го връща като string

def extract_text(path_to_pdf: Path) -> str:
  with PdfReader(path_to_pdf) as pdf_reader:
    return "\n".join(page.extract_text() for page in pdf_reader.pages)

# Разделяне на целия .pdf на парчета - по брой символи, по изречения

In [74]:
# функция, на която подаваме текст, брой символи на парчето и брой символи на застъпване - брой символи в края и в началото на парчето,
# които да се припокриват със следващото парче, тъй като парчетата са по брой символи, то тогава може дума да бъде разделена между 2 парчета
# и за да не изпускаме смисъла на тази дума вземаме по 200 букви от началото и края на парчето

def chunk_by_length(text: str, length: int = 1000, overlap: int = 200) -> List[str]:
  chunks = []
  index = 0

  while index < len(text):
    end = index + length
    current_chunk = text[index:end]
    chunks.append(current_chunk)
    index = end - overlap

  return chunks


In [75]:
# функция, която разделя текста на изречения -> всяко парче, ще е отделно изречение
# има ограничение при над 1000 изречения =>
# BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'input': array length must be 2048 or less.", 'type': 'invalid_request_error', 'param': None, 'code': None}} in add.

def chunk_by_sentences(text: str) -> List[str]:
  sentences = []

  sentences = re.split(r'(?<=[.!?])\s+', text)

  return sentences


# Попълване на векторна база данни

In [76]:
# функция, която пълни векторната база chromadb
# Съвет: векторната база да се пълни на batch-ове, защото при над 1000 записа, chromadb ще се опита да изгради вектори за всички записи едновременно
# и може да достигне "лимити", които да попречат завършване на записване в DB

def add_to_chromadb(ids: List[str], chunks: List[str]) -> None:
  batch_index = 0
  batch_lenght = 100

  while batch_index < len(chunks):
    end = batch_index + batch_lenght

    chroma_collection.add(
      ids = ids[batch_index:end],
      documents = chunks[batch_index: end]
    )
    batch_index = end


In [77]:
class QueryArgs(BaseModel):
    query: str = Field(description="Въпрос на естествен език.")

QueryArgs.model_json_schema()

{'properties': {'query': {'description': 'Въпрос на естествен език.',
   'title': 'Query',
   'type': 'string'}},
 'required': ['query'],
 'title': 'QueryArgs',
 'type': 'object'}

In [78]:
list(QueryArgs.model_fields.keys())

['query']

In [79]:
TOOLS = [
    {
        "type": "function",
        "name": "query",
        "description": "Използвайте този инструмент, за да получавате отговори на въпроси въз основа на съдържанието на базата от знания.",
        "parameters": {
            "type": "object",
            "properties": QueryArgs.model_json_schema()['properties'],
            "required": list(QueryArgs.model_fields.keys()),
            "additionalProperties": False
        },
        "strict": True
    }
]


In [80]:
def query_handler(raw_args: str) -> str:
    query_args = QueryArgs.model_validate(json.loads(raw_args))
    query_result = chroma_collection.query(query_texts=[query_args.query], n_results=3)

    return "\n-----\n".join(doc for doc in query_result['documents'][0])


HANDLERS = {
    "query": query_handler
}

In [81]:
def extract_tool_calls(response: Response):
    return [output_item for output_item in response.output if output_item.type == 'function_call']

In [82]:
# функция, която връща всички ToolCalls

def extract_tool_calls(response: Response):
    return [output_item for output_item in response.output if output_item.type == 'function_call']

In [83]:
pdf_content = extract_text("/content/Ivan_Vazov_-_Nemili-nedragi.pdf")

In [84]:
chunks = chunk_by_length(pdf_content)
len(chunks)

198

In [85]:
sentences = chunk_by_sentences(pdf_content)
sentences_ids = [f"{i + 1}" for i in range(len(sentences))]   # генерираме ids, за да създадем уникални записи в DB с поредния номер на sentence

In [86]:
chunks_ids = [f"{i + 1}" for i in range(len(chunks))]   # генерираме ids, за да създадем уникални записи в DB с поредния номер на chunk
documents = [chunk for chunk in chunks]

In [87]:
add_to_chromadb(ids=chunks_ids, chunks=documents)

In [90]:
def retrieve_information(prompt: str, verbose: bool = True) -> str:
  conversation =[
      {"role": "system", "content": "You are a helpful assistant that should ask questions about the underlying knowledgebase. Use the \"query\" tool aggresively with various arguments to achive best results."},
      {"role": "user", "content": prompt}
  ]

  ai_args = { "model": "gpt-6-luna", "tools": TOOLS, "reasoning": { "effort": "high"} }
  response = openai_client.responses.create(**ai_args, input=conversation)
  tool_calls = extract_tool_calls(response)

  while tool_calls:
    conversation.extend(response.output)

    if verbose:
      print(f"Found {len(tool_calls)} tools to execute.")

    for tc in tool_calls:
      handler = HANDLERS[tc.name]
      result = handler(tc.arguments)

      conversation.append({"type": "function_call_output", "call_id": tc.call_id, "output": result})

    response = openai_client.responses.create(**ai_args, input=conversation)
    tool_calls = extract_tool_calls(response)

  return response.output_text

In [91]:
retrieve_information("смърт")

Found 4 tools to execute.


'В намерените откъси от „Немили-недраги“ смъртта е свързана най-вече със Странджата: той е тежко болен, приема, че умира, и съжалява, че смъртта го заварва далеч от борбата. За хъшовете смъртта в името на свободата е представена като достойна саможертва.\n\nИскате ли да разгледаме **темата за смъртта в творбата**, **смъртта на Странджата** или значението на думата?'